# Run-book 214-P6 — lineage of a fused training set and its inference, on Azure ML

**Proves:** AC8: `lineage.json` for a full-year (T = 12) fused training set and for its inference on Azure ML, with the training lineage carried through a bundle.
**Needs:** an Azure login (`rise_eo.config.load()`); an AML image built with `runbooks/build-test-images.ipynb` and its commit (both in Settings); the blob prefix of the 214-P5 run-book. Download tens of minutes to hours, two AML jobs minutes each (estimates); about 50 MB on this laptop. Sentinel-1 never downloads here.

Run it cell by cell. Paste back the **Summary** output, or the failing cell's message.

## Before you start

- **Kernel:** `.venv` from the main checkout (`rise-eo/.venv/bin/python`). Setup checks it.
- **Resume:** after a restart, run Setup and Settings, then any step.
- **When done:** Restart & Clear All Outputs, then save (a test refuses saved outputs).
- **Resume:** downloads skip files already complete.

In [ ]:
# Setup -- nothing to change here.
import sys
from pathlib import Path

VENV = Path(sys.prefix)
assert VENV.name == ".venv" and (VENV.parent / "pyproject.toml").exists(), (
    f"Wrong kernel: {sys.prefix}. Select Kernel -> Python Environments -> the main checkout's .venv.")
MAIN_CHECKOUT = VENV.parent

# Import this notebook's own src/, not the main checkout's (matters in a worktree).
CHECKOUT = Path.cwd().parent
assert Path.cwd().name == "runbooks" and (CHECKOUT / "src" / "rise_eo").is_dir(), (
    f"The kernel started in {Path.cwd()}, not in this notebook's runbooks/ folder.")
if str(CHECKOUT / "src") not in sys.path:
    sys.path.insert(0, str(CHECKOUT / "src"))
import rise_eo

assert Path(rise_eo.__file__).is_relative_to(CHECKOUT / "src"), (
    f"rise-eo was already imported from {rise_eo.__file__}. Restart the kernel, then run this cell first.")
print("rise-eo:", rise_eo.__file__)

## Settings

In [ ]:
# Settings
import datetime
import json

import geopandas as gpd
import numpy as np
import rasterio.features

import rise_eo
from rise_eo.storage import fs
from rise_eo.storage.azure import configure_storage

cfg = rise_eo.config.load()      # subscription, resource group, workspace, cluster, uami: private, read at run time
configure_storage("azure")       # so every step can read blob, also after a kernel restart

# Replace <your-user> with your own blob prefix before running; a forgotten placeholder fails below.
AZ_ROOT_P5 = "abfss://data@strisewesteurope.dfs.core.windows.net/<your-user>/214-p5-fused"   # the AC7 run-book's prefix
AZ_ROOT = "abfss://data@strisewesteurope.dfs.core.windows.net/<your-user>/214-p6-lineage"
assert "<" not in AZ_ROOT_P5 + AZ_ROOT, "set your own blob prefix in AZ_ROOT_P5 and AZ_ROOT (replace <your-user>)"
# The AML environment built from main after this PR merged ("<name>:<version>"), and the commit it was built from.
AZ_ENV = "<environment name>:<version>"
IMAGE_COMMIT = "<commit>"
assert "<" not in AZ_ENV + IMAGE_COMMIT, "set AZ_ENV and IMAGE_COMMIT (replace the placeholders)"

ARCHIVE = f"{AZ_ROOT_P5}/imagery"        # the AC7 archive: this run downloads the rest of the year into it
RUN_ROOT = f"{AZ_ROOT}/run"              # blob working root: cubes, input.csv, the raw flatten output
RUN_FOLDER = f"{RUN_ROOT}/runs/train"    # where create_training_data puts the per-field cubes
INFER = f"{AZ_ROOT}/inference"           # run_inference's output folder: cells, COGs, STAC, lineage.json
INFER_REFUSED = f"{AZ_ROOT}/inference-refused"
OUT = MAIN_CHECKOUT / "tests" / "outputs" / "214_p6_lineage"   # gitignored: landed arrays, the bundle, the lineage files
BUNDLE = str(OUT / "bundle")

SHAPES = CHECKOUT / "notebooks" / "shapefiles"
ROI = SHAPES / "s2grid=4772924.geojson"          # inside MGRS tile T33UWP, with labels
LABELS = SHAPES / "AT_2018_TRAIN.geojson"        # 900 EuroCrops fields, columns fid and crop
START = datetime.datetime(2018, 1, 1)            # one calendar year: T = 12
END = datetime.datetime(2019, 1, 1)
MOSAIC_DAYS = "month"
T = 12
MAX_TILES = 400                                  # per collection; the download refuses a larger match

S2 = rise_eo.CollectionInput(
    "sentinel-2-l2a",
    ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12", "SCL"],
    processing="latest",                          # one processing per acquisition
)
S1 = rise_eo.CollectionInput(
    "sentinel-1-rtc", ["vv", "vh"],
    properties_filter={"sat:orbit_state": ["ascending", "descending"]},   # opt in to median both orbits
)
S1_ONE_ORBIT = rise_eo.CollectionInput(
    "sentinel-1-rtc", ["vv", "vh"], properties_filter={"sat:orbit_state": "descending"},
)

runner_kwargs = dict(
    cluster=cfg.cluster, environment=AZ_ENV, root=RUN_ROOT,
    identity_client_id=cfg.uami_client_id,        # the NODES' identity, not your login
    subscription_id=cfg.subscription_id, resource_group_name=cfg.resource_group,
    workspace_name=cfg.workspace, poll_interval_seconds=10,
)
OUT.mkdir(parents=True, exist_ok=True)

## Helpers

Run after Settings on every restart.

In [ ]:
# Helpers
FUSED = r"s2l2a-s1rtc_20180101-20190101_P1M_[0-9a-f]{8}"


def segment(folder):
    """The one fused segment under `folder` on blob."""
    import re

    found = [p.rstrip("/").split("/")[-1] for p in fs.ls(folder)]
    hits = [s for s in found if re.fullmatch(FUSED, s)]
    assert len(hits) == 1, f"expected one segment matching {FUSED} in {folder}, found {found}"
    return hits[0]


def landed(name):
    """One landed training array, read from this laptop."""
    return np.load(OUT / "training" / f"{name}.npy")


def cube(folder):
    """`(datacube, metadata)` of the cube in `folder` on blob."""
    return (fs.load_npy(f"{folder}/datacube.npy"),
            fs.load_npy(f"{folder}/metadata.pickle.npy", allow_pickle=True)[()])


def label_fields():
    """The `AT_2018_TRAIN` fields inside the cell, in the cell's CRS."""
    cell = gpd.read_file(ROI)
    fields = gpd.read_file(LABELS).to_crs(cell.crs)
    return fields[fields.intersects(cell.geometry.iloc[0])]


def read_json(path):
    with fs.open(path, "r") as f:
        return json.load(f)


def show(doc):
    """A lineage document with each granule id list reduced to its length, for reading."""
    doc = json.loads(json.dumps(doc))
    for c in doc.get("sources", {}).get("collections", []):
        c["granule_ids"] = f"<{len(c['granule_ids'])} ids>"
    if doc.get("training_lineage"):
        doc["training_lineage"] = show(doc["training_lineage"])
    return json.dumps(doc, indent=2)


def check_lineage_common(doc):
    """What both lineage files must say: the version, no private identifier, and the AML image's commit."""
    assert doc["lineage_version"] == 1
    text = json.dumps(doc)
    for private in (cfg.subscription_id, cfg.resource_group, cfg.workspace, cfg.cluster, cfg.uami_client_id):
        assert not private or private not in text, "a private identifier is in lineage.json"
    assert "runner_kwargs" not in doc["request"]["arguments"]
    builds = doc["code"]["build"]
    assert builds, "code.build is empty"
    assert {b.get("commit") for b in builds} == {IMAGE_COMMIT}, (
        f"code.build names {[b.get('commit') for b in builds]}, not the AML image's commit {IMAGE_COMMIT}")
    assert not any(b.get("dirty") for b in builds), builds

## Step 1 — download the rest of the year (AC4)

Downloads the other months into the 214-P5 archive. **PASS:** both catalogs exist on blob. **Time:** the longest step.

In [ ]:
# Step 1 -- download.
catalogs = rise_eo.download(
    roi=str(ROI), startdate=START, enddate=END, bands=None, dst_folderpath=ARCHIVE,
    inputs=[S2, S1], source="mpc", max_tiles=MAX_TILES, storage="azure",
    runner="aml", runner_kwargs=runner_kwargs,
)
print(json.dumps(catalogs, indent=2))
for collection in (S2.collection, S1.collection):
    expected = rise_eo.archive_catalog_filepath(ARCHIVE, collection)
    assert fs.exists(expected), f"no catalog for {collection} at {expected}"
print("both catalogs are on blob")

## Step 2 — build the year's fused cubes and flatten them (AC8)

Builds and flattens the year's fused cubes on AML. **PASS:** `training.n_timestamps` is 12 and the bands are the 12 Sentinel-2 bands then `vv`, `vh`. **Time:** two AML jobs.

In [ ]:
# Step 2 -- build and flatten.
fields = label_fields()
print(f"{len(fields)} fields inside {ROI.name}")

training = rise_eo.create_training_data(
    label_polygons=fields, catalog_filepath=None, startdate=START, enddate=END,
    mosaic_days=MOSAIC_DAYS, bands=None, id_col="fid", label_col="crop",
    inputs=[S2, S1], archive_root=ARCHIVE,
    export_folderpath=str(OUT / "training"), run_folderpath=RUN_FOLDER,
    runner="aml", runner_kwargs=runner_kwargs,
)
S2_OUT = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12"]
assert training.n_timestamps == T, training.n_timestamps
assert training.bands == S2_OUT + ["vv", "vh"], training.bands
print(training.export_folderpath, training.n_pixels, "pixels, T =", training.n_timestamps)

## Step 3 — the training `lineage.json` (D8)

**PASS:** steps are download, mask, resample, mosaic, fuse, flatten; both collections list granules from their catalogs; the geometry sha256 matches; no private identifiers; `code.build` is the AML image's commit, clean. Read the printed file.

In [ ]:
# Step 3 -- check it (D8).
import hashlib

lineage = json.loads((OUT / "training" / "lineage.json").read_text())
check_lineage_common(lineage)
assert [s["step"] for s in lineage["process_steps"]] == [
    "download", "mask", "resample", "mosaic", "fuse", "flatten"], [s["step"] for s in lineage["process_steps"]]
assert f"T={T}" in lineage["statement"] and S2.collection in lineage["statement"] and S1.collection in lineage["statement"]
assert lineage["code"]["driver"]["rise-eo"] == lineage["software"]["rise-eo"]

for entry, collection in zip(lineage["sources"]["collections"], (S2, S1)):
    assert entry["collection"] == collection.collection
    catalog_ids = set(fs.read_parquet(rise_eo.archive_catalog_filepath(ARCHIVE, collection.collection))["id"])
    assert entry["n_granules"] == len(entry["granule_ids"]) > 0, entry["collection"]
    assert set(entry["granule_ids"]) <= catalog_ids, entry["collection"]

geometry = lineage["sources"]["geometry"]
assert hashlib.sha256((OUT / "training" / "geometry.geojson").read_bytes()).hexdigest() == geometry["sha256"]
assert geometry["n_features"] == len(label_fields())
print(show(lineage))

## Step 4 — a bundle that carries the training lineage (D9)

Saves a stub adapter as a bundle. **PASS:** the bundle's lineage equals Step 3's file.

In [ ]:
# Step 4 -- the stub adapter and its bundle.
import importlib
import sys

STUB = OUT / "stub"
STUB.mkdir(exist_ok=True)
(STUB / "stub_adapter.py").write_text('''
import numpy as np

from rise_eo.model import BaseModelAdapter


class StubAdapter(BaseModelAdapter):
    required_bands = ["B08", "vv"]
    n_timestamps = 12
    output_dtype = "uint8"
    output_nodata = 255
    output_band_names = ["class"]
    feature_sequence = []

    def load(self):
        pass

    def predict(self, X_chunk):
        return np.zeros(len(X_chunk), dtype="uint8")
''')
sys.path.insert(0, str(STUB))
stub_adapter = importlib.import_module("stub_adapter")

rise_eo.save_bundle(stub_adapter.StubAdapter(), {}, BUNDLE, training_data=str(OUT / "training"))
manifest = json.loads((OUT / "bundle" / "bundle.json").read_text())
assert manifest["training_lineage"] == "lineage.json", manifest
assert json.loads((OUT / "bundle" / "lineage.json").read_text()) == lineage
print("the bundle carries the training lineage")

## Step 5 — a one-orbit request is refused before any job starts (D9, AC11)

Calls `run_inference` with Sentinel-1 limited to `descending`. **PASS:** it raises `PreflightError` naming `properties_filter`, and writes nothing.

In [ ]:
# Step 5 -- check it (D9).
try:
    rise_eo.run_inference(
        BUNDLE, roi=str(ROI), inputs=[S2, S1_ONE_ORBIT], archive_root=ARCHIVE,
        startdate=START, enddate=END, mosaic_days=MOSAIC_DAYS,
        output_folderpath=INFER_REFUSED, runner="aml", storage="azure", runner_kwargs=runner_kwargs,
    )
except rise_eo.PreflightError as exc:
    refusal = str(exc)
else:
    raise AssertionError("run_inference accepted a request the model was not trained on")
print(refusal)
assert "properties_filter" in refusal and "descending" in refusal and "ascending" in refusal, refusal
assert not fs.exists(f"{INFER_REFUSED}/lineage.json")

## Step 6 — inference over the cell with the same inputs (D6, D9)

Runs inference on AML. **PASS:** the call returns COGs and a STAC catalog. **Time:** one build and inference job.

In [ ]:
# Step 6 -- inference.
result = rise_eo.run_inference(
    BUNDLE, roi=str(ROI), inputs=[S2, S1], archive_root=ARCHIVE,
    startdate=START, enddate=END, mosaic_days=MOSAIC_DAYS,
    output_folderpath=INFER, runner="aml", storage="azure", runner_kwargs=runner_kwargs,
)
print(len(result.output_filepaths), "grid cells inferred; STAC:", result.stac_catalog_filepath)
assert result.output_filepaths

## Step 7 — the inference cube equals the training cube on a field's pixels (D6)

**PASS:** same segment name, grids aligned, every pixel inside one field identical.

In [ ]:
# Step 7 -- check it (D6).
grids = gpd.GeoDataFrame.from_features(read_json(f"{INFER}/grids.geojson")["features"], crs="EPSG:4326")
train_segment = segment(RUN_FOLDER)
infer_segment = segment(f"{INFER}/cells")
assert train_segment == infer_segment, (train_segment, infer_segment)

fields_ll = label_fields().to_crs("EPSG:4326")
FIELD = CELL = None
for fid in np.unique(landed("ids")):
    geometry = fields_ll[fields_ll["fid"].astype(str) == str(fid)].geometry.iloc[0]
    inside = grids[grids.geometry.contains(geometry)]
    if len(inside):
        FIELD, CELL = str(fid), str(inside["id"].iloc[0])
        break
assert FIELD is not None, "no training field lies inside one grid cell"
print("field", FIELD, "in grid cell", CELL)

train_cube, train_md = cube(f"{RUN_FOLDER}/{train_segment}/{FIELD}")
infer_cube, infer_md = cube(f"{INFER}/cells/{infer_segment}/{CELL}")
gt_t, gt_i = train_md["geotiff_metadata"], infer_md["geotiff_metadata"]
assert gt_t["crs"] == gt_i["crs"] and gt_t["transform"].a == gt_i["transform"].a
dx = (gt_t["transform"].c - gt_i["transform"].c) / gt_i["transform"].a
dy = (gt_t["transform"].f - gt_i["transform"].f) / gt_i["transform"].e
assert abs(dx - round(dx)) < 1e-6 and abs(dy - round(dy)) < 1e-6, (dx, dy)
dx, dy = round(dx), round(dy)

shape = gpd.GeoSeries(train_md["geometry"]["shape"], crs=train_md["geometry"]["crs"]).to_crs(gt_t["crs"])
outside = rasterio.features.geometry_mask(shape, out_shape=(gt_t["height"], gt_t["width"]),
                                          transform=gt_t["transform"], all_touched=True)
rows, cols = np.nonzero(~outside)
assert train_cube.shape[0] == infer_cube.shape[0] == T and train_cube.shape[-1] == infer_cube.shape[-1] == 14
a = train_cube[:, rows, cols, :]
b = infer_cube[:, rows + dy, cols + dx, :]
assert np.array_equal(a, b), f"{int((a != b).sum())} of {a.size} values differ"
print(f"{len(rows)} field pixels x {T} months x 14 bands are identical in the training and inference cubes")

## Step 8 — the inference `lineage.json` and the STAC properties (D8)

**PASS:** steps end `fuse`, `predict`; the model digest is recorded; the training lineage equals Step 3's; the STAC Collection links `lineage.json`. Read the printed file.

In [ ]:
# Step 8 -- check it (D8).
infer_lineage = read_json(f"{INFER}/lineage.json")
check_lineage_common(infer_lineage)
assert [s["step"] for s in infer_lineage["process_steps"]] == [
    "download", "mask", "resample", "mosaic", "fuse", "predict"], [s["step"] for s in infer_lineage["process_steps"]]
assert infer_lineage["model"]["digest"].startswith("sha256:"), infer_lineage["model"]
assert infer_lineage["training_lineage"] == lineage
outputs = fs.glob(f"{INFER}/cells/*/*/output.tif")
assert infer_lineage["sources"]["grid"]["n_cells"] == len(outputs) > 0

stac_collection = read_json(f"{INFER}/stac/rise-eo-inference/collection.json")
assert stac_collection["processing:software"]["rise-eo"] == infer_lineage["software"]["rise-eo"]
assert stac_collection["processing:lineage"] == infer_lineage["statement"]
asset = stac_collection["assets"]["lineage"]
assert asset["roles"] == ["metadata"] and asset["href"].endswith("lineage.json"), asset
print(show(infer_lineage))
(OUT / "inference_lineage.json").write_text(json.dumps(infer_lineage, indent=2))

## Summary — paste this back

In [ ]:
lineage = json.loads((OUT / "training" / "lineage.json").read_text())
infer_lineage = read_json(f"{INFER}/lineage.json")
summary = {
    "training_steps": [s["step"] for s in lineage["process_steps"]],
    "training_granules": {c["collection"]: c["n_granules"] for c in lineage["sources"]["collections"]},
    "training_build_code": lineage["code"]["build"],
    "training_driver_code": lineage["code"]["driver"],
    "overlapping_mgrs_tiles": lineage["determinism"]["overlapping_mgrs_tiles"],
    "inference_steps": [s["step"] for s in infer_lineage["process_steps"]],
    "inference_cells": infer_lineage["sources"]["grid"]["n_cells"],
    "model_digest": infer_lineage["model"]["digest"],
    "inference_build_code": infer_lineage["code"]["build"],
    "files": [str(OUT / "training" / "lineage.json"), str(OUT / "inference_lineage.json")],
}
print(json.dumps(summary, indent=2, default=str))